In [ ]:
!pip install deepxde torch numpy matplotlib

In [ ]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# --- Config ---
ALPHA      = 1.0
T_MAX      = 0.01
N_DOMAIN   = 10000
N_BOUNDARY = 800
N_INITIAL  = 500
W_PDE = 1
W_BC  = 10
W_IC  = 10
T_EVAL = [0.001, 0.005, 0.010]

# Fourier feature config
SIGMA_BANDS = [1, 5, 20]   # frequency band scales
M_FREQ      = 64           # random frequencies per band
# Total Fourier features: len(SIGMA_BANDS) * 2 * M_FREQ = 384

print(f"Backend: {dde.backend.backend_name}")
print(f"T_MAX={T_MAX}")
print(f"Fourier features: {len(SIGMA_BANDS)} bands × 2 × {M_FREQ} = {len(SIGMA_BANDS)*2*M_FREQ}")
print("Config loaded.")

In [ ]:
def T_exact(x, y, t):
    """
    Two-mode analytical solution:
      mode 1 (low freq):  sin(πx)sin(πy) · exp(−2π²t)
      mode 2 (high freq): 0.3·sin(5πx)sin(5πy) · exp(−50π²t)
    """
    low  = np.sin(np.pi * x) * np.sin(np.pi * y) * np.exp(-2  * np.pi**2 * ALPHA * t)
    high = 0.3 * np.sin(5 * np.pi * x) * np.sin(5 * np.pi * y) * np.exp(-50 * np.pi**2 * ALPHA * t)
    return low + high

# Sanity checks
# 1. At t=0, T_exact should equal the IC
val = T_exact(np.array([0.5]), np.array([0.1]), 0.0)
ic  = np.sin(np.pi*0.5)*np.sin(np.pi*0.1) + 0.3*np.sin(5*np.pi*0.5)*np.sin(5*np.pi*0.1)
assert np.isclose(val[0], ic), f"IC mismatch: {val[0]} vs {ic}"
print(f"IC sanity check passed: T_exact(0.5, 0.1, 0) = {val[0]:.6f}")

# 2. Show high-freq mode amplitude at eval times
print("\nHigh-freq mode amplitude at eval times:")
for t in T_EVAL:
    amp = 0.3 * np.exp(-50 * np.pi**2 * ALPHA * t)
    print(f"  t={t:.3f}: {amp:.4f}")

In [ ]:
geom       = dde.geometry.Rectangle(xmin=[0, 0], xmax=[1, 1])
timedomain = dde.geometry.TimeDomain(0, T_MAX)
geomtime   = dde.geometry.GeometryXTime(geom, timedomain)

print(f"Domain: [0,1]² × [0, {T_MAX}]")
print("GeometryXTime created successfully.")

In [ ]:
def pde(x, u):
    """
    Residual: ∂u/∂t − α(∂²u/∂x² + ∂²u/∂y²) = 0
    x[:,0]=x_coord, x[:,1]=y_coord, x[:,2]=t
    """
    u_t  = dde.grad.jacobian(u, x, i=0, j=2)
    u_xx = dde.grad.hessian(u, x, component=0, i=0, j=0)
    u_yy = dde.grad.hessian(u, x, component=0, i=1, j=1)
    return u_t - ALPHA * (u_xx + u_yy)

print("PDE residual function defined.")

In [ ]:
# Dirichlet BC: T = 0 on all boundaries
bc = dde.icbc.DirichletBC(
    geomtime,
    func=lambda x: np.zeros((len(x), 1)),
    on_boundary=lambda x, on_boundary: on_boundary,
)

# IC: T(x,y,0) = sin(πx)sin(πy) + 0.3·sin(5πx)sin(5πy)
def ic_func(x):
    low  = np.sin(np.pi * x[:, 0:1]) * np.sin(np.pi * x[:, 1:2])
    high = 0.3 * np.sin(5 * np.pi * x[:, 0:1]) * np.sin(5 * np.pi * x[:, 1:2])
    return low + high

ic = dde.icbc.IC(
    geomtime,
    func=ic_func,
    on_initial=lambda x, on_initial: on_initial,
)

print("BC (Dirichlet, T=0) defined.")
print("IC (two-mode sinusoidal) defined.")

In [ ]:
def solution_func(x):
    """Analytical solution for DeepXDE test metric."""
    low  = (np.sin(np.pi * x[:, 0:1]) * np.sin(np.pi * x[:, 1:2])
            * np.exp(-2 * np.pi**2 * ALPHA * x[:, 2:3]))
    high = (0.3 * np.sin(5 * np.pi * x[:, 0:1]) * np.sin(5 * np.pi * x[:, 1:2])
            * np.exp(-50 * np.pi**2 * ALPHA * x[:, 2:3]))
    return low + high

data = dde.data.TimePDE(
    geometryxtime=geomtime,
    pde=pde,
    ic_bcs=[bc, ic],
    num_domain=N_DOMAIN,
    num_boundary=N_BOUNDARY,
    num_initial=N_INITIAL,
    train_distribution="LHS",
    solution=solution_func,
    num_test=1000,
)

print("Dataset assembled:")
print(f"  Domain : {N_DOMAIN} pts (LHS)")
print(f"  Boundary: {N_BOUNDARY} pts")
print(f"  IC      : {N_INITIAL} pts")
print(f"  Test    : 1000 pts vs analytical solution")